In [ ]:
import os
import numpy as np
import pandas as pd


import pickle
from pathlib import Path

In [ ]:
results = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/results/2025-11-18_get_condition_vaccines"
data = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/data/2025-11-18_get_condition_vaccines"

data1 = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/data/2025-11-13_mapping_condition_cohorts_to_viral_specie"
data2 ="/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/data/2025-11-13_get_cohort_statistic"

In [ ]:
## 1. get data

In [ ]:
def get_data_pkl(path, filename):

    # …later, in any notebook in the same workspace…
    out_file = Path(f'{path}/{filename}')

    # Reload:
    with open(out_file, 'rb') as f:
        cohort_data_dict = pickle.load(f)

    #print("Reloaded keys:", list(cohort_data_dict.keys()))
    
    return cohort_data_dict

In [ ]:
## 2. wrangle data

In [ ]:
def wrangle_annot_ns_vax_df():
    
    
    viral_concept_df = pd.read_csv(f'{data1}/viral_cohort_person_id_overlap_annotated.csv')
    
    df1 = viral_concept_df.copy()
    
    nonseasonal_table = df1.loc[:, ['condition_concept_id','standard_concept_name','non_seasonal_vax',  'specie', 'genus'] ]
    ns_vax = nonseasonal_table.loc[nonseasonal_table['non_seasonal_vax'] == 'Y', :]
    
    
    def merge_specie_data(new_column):

        if new_column["genus"] == "Influenzavirus":
            return new_column["genus"]
        else:
            return new_column["specie"]


    ns_vax_table = ns_vax.copy()
    
    ns_vax_table["updated_specie"] = ns_vax_table.apply(merge_specie_data, axis=1)
    ns_table = ns_vax_table.loc[:, ['condition_concept_id','standard_concept_name', 'updated_specie']]
    ns_table

    return ns_table

In [ ]:
def get_concept_vax_id_dict(): 
    '''
       1. subset a new df with only 2 columns: concept_id and vaccine_id
       2. reset the index to concept_id
       3. groupby concept_id, selecting (vaccine_id) col as a series, and using .agg to apply an aggregation to each 
        group, that being the lambda function to make the grouped series into a single list
    
    ''' 
    
    #mapping specie:vaccine list
    vaccine_concepts_df = pd.read_csv(f'{data}/ns_viral_cohort_vax_ids.csv').loc[:, ['updated_specie', 'vaccine_id']]
    vaccine_concepts_df
    
    ns_cohort_vaccine_df = ns_vax_cohorts.merge(vaccine_concepts_df,  on='updated_specie', how='left')
    ns_cohort_vaccine_df
    
    df1 = ns_cohort_vaccine_df.loc[ :, ['condition_concept_id', 'standard_concept_name', 'vaccine_id']]
    df2 = df1.set_index(['condition_concept_id', 'standard_concept_name'])
    df3 = df2.groupby(['condition_concept_id', 'standard_concept_name'], sort=False)['vaccine_id'].agg(lambda vax_id: list(vax_id))
    final = df3.to_dict()
    
    
    return final

In [ ]:
#wrangle cond, vax, and demo dfs

In [ ]:
def get_cond_cohort_df_dict(all_cond_df): 
    print("Grouping DataFrame into a dictionary...")

    # Define the columns you want to use for your composite key
    key_columns = ['condition_concept_id', 'standard_concept_name']

    # Use a dictionary comprehension with groupby to create the dictionary
    # - The 'key' will be a tuple: (condition_concept_id, standard_concept_name)
    # - The 'group_df' will be the DataFrame containing all rows for that key
    concept_groups_dict = {
        key: group_df 
        for key, group_df in all_cond_df.groupby(key_columns)
    }

    print(f"Successfully created a dictionary with {len(concept_groups_dict)} unique (ID, Name) keys.")
    
    return concept_groups_dict

In [ ]:
def filter_cond_df_dict(cohort_dict):
    
    
    master_df_of_final_cohorts = ns_vax_cohorts.copy()
    
    master_key_cols = ['condition_concept_id', 'standard_concept_name']

    # Create a set of tuples (key1, key2) from the master DataFrame.
    # This set will act as our "allow list".
    valid_keys_set = set(
        master_df_of_final_cohorts[master_key_cols].itertuples(index=False, name=None)
    )
    
    # 'concept_groups_dict' is the dictionary you created in the previous step
    # 'valid_keys_set' is the set we just created

    filtered_concept_dict = {
        key: group_df 
        for key, group_df in cohort_dict.items() 
        if key in valid_keys_set
    }

    print(f"Original dictionary had {len(cohort_dict)} items.")
    print(f"Filtered dictionary now has {len(filtered_concept_dict)} items.")

    # You can now work with your new, smaller dictionary
    # print(list(filtered_concept_dict.keys())[:5])
    
    return filtered_concept_dict

In [ ]:
def wrangle_cohort_df_dict(filtered_cohort_dict): 

    cohort_dict = {}
        
    for key, df in filtered_cohort_dict.items():

        # 2. Sort by person_id and date so 'first' and 'last' work correctly
        df = df.sort_values(by=['person_id', 'condition_start_datetime'])

        # 3. Group by person_id and aggregate the data
        #    - 'first' gets the earliest diagnosis
        #    - 'last' gets the latest diagnosis
        #    - 'nunique' counts the number of unique visit IDs
        #    - 'size' counts the total number of rows (unique inputs)
        result = df.groupby('person_id').agg(
            condition_concept_id =('condition_concept_id', "first"),
            standard_concept_name=('standard_concept_name', "first"),
            first_diagnosis_date=('condition_start_datetime', 'first'),
            last_diagnosis_date=('condition_start_datetime', 'last'),
            number_of_visit_occurences=('visit_occurrence_id', 'nunique'),  # Counts unique visit IDs
            total_diagnosis_of_concept_id=('condition_start_datetime', 'nunique')                    # Counts total rows for the person
        ).reset_index()

        
        # 3. Logic: Blank out Last Date if it equals First Date
        result['last_diagnosis_date'] = np.where(
            result['first_diagnosis_date'] == result['last_diagnosis_date'], 
            pd.NaT, 
            result['last_diagnosis_date']
        )
       
    # Display the result
        cohort_dict[key] = result
    
    return cohort_dict


In [ ]:
def get_vax_cohort_df_dict(all_vax_df, concept_vax_id_dict): 
    # Initialize the dictionary to hold the resulting DataFrames
    individual_cohort_vax_dfs = {}

    # Iterate through the dictionary
    for cohort_key, vaccine_concept_list in concept_vax_id_dict.items():

        # cohort_key is the tuple (concept_id, concept_name)

        # Use .isin() to create a boolean mask for the drug_concept_id column
        mask = all_vax_df['drug_concept_id'].isin(vaccine_concept_list)

        # Filter the master DF using the mask
        filtered_df = all_vax_df[mask].copy()

        # Store the filtered DataFrame in the new dictionary, using the tuple as the key
        individual_cohort_vax_dfs[cohort_key] = filtered_df
        
    return individual_cohort_vax_dfs

In [ ]:
def merge_vax_dict_w_cond_dict(dict_A, dict_B):

    merged_data_dict = {}
       # Only keep IDs present in both DFs (change to 'left' if needed)

    # Iterate through the keys of the first dictionary (dict_A)
    for cohort_name, df_a in dict_A.items():

        # Check if the same key exists in the second dictionary (dict_B)
        if cohort_name in dict_B:

            # Get the corresponding DataFrame from dict_B
            df_b = dict_B[cohort_name]

            print(f"Merging cohort: **{cohort_name}**")

            # Perform the merge operation
            merged_df = pd.merge(
                left=df_a,
                right=df_b,
                on='person_id',
                how= 'right'
            ) 
            
            demo_merged_df = pd.merge(
                left=demo,
                right=merged_df,
                on='person_id',
                how= 'right'
            ) 
            # Store the resulting merged DataFrame in the new dictionary
            merged_data_dict[cohort_name] = demo_merged_df

        else:
            print(f"Skipping cohort {cohort_name}: Key not found in dict_B.")

    return merged_data_dict



In [ ]:
def get_vax_status_df():  
    new_dict = {}

    for key, df in final_vax_cond_df.items():


        df1 = df.drop_duplicates(subset="person_id", keep="first")

        vax_df = df.loc[df['drug_exposure_start_datetime']  < df['first_diagnosis_date']].copy()

        vax_df = (
                vax_df
                .sort_values("drug_exposure_start_datetime")
                .drop_duplicates(subset="person_id", keep="first"))


        final_merge = df1.merge(vax_df, on = 'person_id', how = 'left' )


        new_dict[key] =  final_merge
        
    return new_dict
    

In [ ]:
def get_ancestry_vax_stats(ns_vax_df):
        
    final_dict = {}
    
    def merge_race_ethnicity_data(new_column):

        if new_column["ethnicity_x"] == "Hispanic or Latino":
            return new_column["ethnicity_x"]
        else:
            return new_column["race_x"]

    def add_vax_status(new_column):

        if new_column["drug_exposure_start_datetime_x"] is pd.NaT:
            return 'N'
        else:
            return 'Y'


    for key, table in ns_vax_df.items():

        table["updated_race"] = table.apply(merge_race_ethnicity_data, axis=1)
        table["vaccinated"] = table.apply(add_vax_status, axis=1)
        
        final_dict[key] = table
    
    
    return final_dict

In [ ]:
def create_df_pkl(df, directory):

    # 2) Choose a workspace folder for persistence
    out_file = Path(directory)

    # 3) Save the entire dict in one go
    with open(out_file, 'wb') as f:
        pickle.dump(df, f)

    print(f"Saved {len(df)} DataFrames to {out_file}")

In [ ]:
## function calls

##get vax dict and cohort csv

ns_vax_cohorts = wrangle_annot_ns_vax_df()
concept_vax_id_dict = get_concept_vax_id_dict()

##get_data_pkl

cohort = get_data_pkl(data2, 'cohort_cond_df.pkl')
demo = get_data_pkl(data2, 'cohort_demo_df.pkl')
vax = get_data_pkl(data, 'cohort_vax_df.pkl')

##wrangle data

cohort_dict = get_cond_cohort_df_dict(cohort)
filtered_cohort_dict = filter_cond_df_dict(cohort_dict)
wrangled_cohort_dict = wrangle_cohort_df_dict(filtered_cohort_dict)
vax_cohort_dict = get_vax_cohort_df_dict(vax, concept_vax_id_dict)

##merge vax, cohort, demo data 


final_vax_cond_df = merge_vax_dict_w_cond_dict(vax_cohort_dict, wrangled_cohort_dict)

vacc_df = get_vax_status_df()

final_df = get_ancestry_vax_stats(vacc_df)

create_df_pkl(final_df, f"{results}/cohort_vaccine_dict.pkl")

In [ ]:
## Visualize

ns_vax_cohorts
#concept_vax_id_dict
#vax

#merged_stats_df_dict

#final_vax_cond_df

#vacc_df


#final_df

In [ ]:
viral_concept_df = pd.read_csv(f'{data1}/viral_cohort_person_id_overlap_annotated.csv')
    

In [ ]:
viral_concept_df